In [ ]:
import os
import re
import random
import timm
import copy
import torchvision
from torchvision import datasets, transforms
import torchvision.models as models
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

def seed_all(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

batch_size = 64
seed = 123
seed_all(seed)

byol_epochs = 200
b_learning_rate = 1e-3


num_class = 3

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Part1.Dataset

from pathlib import Path
from typing import List, Sequence, Optional
from PIL import Image, ImageOps
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import functional as TF

class Clsdata(Dataset):
    def __init__(
        self,
        root_list: Sequence[str], # front images path list
        cls_to_idx,
        shuffle: bool = True,
        transform=None,
        train: bool = False,
        test: bool = False,
        img_size: int = 224,
    ):
        assert not (train and test), "Both train and test cannot be True at the same time"
        self.test = test
        self.train = train
        self.img_size = int(img_size)

        lines: List[Path] = [Path(p) for p in root_list]
        if shuffle:
            random.shuffle(lines)
        self.lines = lines
        self.nSamples = len(self.lines)

        self.mean = (0.485, 0.456, 0.406)
        self.std  = (0.229, 0.224, 0.225)

        if transform is None:
            if self.train and not self.test:
                self.transform = None
            else:
                # val / test：
                self.transform = transforms.Compose([
                    transforms.Resize((self.img_size, self.img_size)),
                    transforms.ToTensor(),
                    transforms.Normalize(self.mean, self.std),
                ])
        else:
            self.transform = transform

        self.cls_to_idx = cls_to_idx
        self.idx_to_cls = {v: k for k, v in self.cls_to_idx.items()}

    def __len__(self):
        return self.nSamples

    def _pair_transform(self, img1_pil: Image.Image, img2_pil: Image.Image):
        img1 = ImageOps.exif_transpose(img1_pil)
        img2 = ImageOps.exif_transpose(img2_pil)

        img1 = TF.resize(img1, [self.img_size, self.img_size])
        img2 = TF.resize(img2, [self.img_size, self.img_size])

        if random.random() < 0.5:
            img1 = TF.hflip(img1)
            img2 = TF.hflip(img2)

        if random.random() < 0.5:
            img1 = TF.vflip(img1)
            img2 = TF.vflip(img2)

        if random.random() < 0.8:
            b = 1.0 + random.uniform(-0.2, 0.2)  # brightness factor
            c = 1.0 + random.uniform(-0.2, 0.2)  # contrast factor
            s = 1.0 + random.uniform(-0.2, 0.2)  # saturation factor
            h = random.uniform(-0.05, 0.05)      # hue delta

            img1 = TF.adjust_brightness(img1, b); img2 = TF.adjust_brightness(img2, b)
            img1 = TF.adjust_contrast(img1, c);   img2 = TF.adjust_contrast(img2, c)
            img1 = TF.adjust_saturation(img1, s); img2 = TF.adjust_saturation(img2, s)
            img1 = TF.adjust_hue(img1, h);        img2 = TF.adjust_hue(img2, h)

        # ToTensor + Normalize
        img1 = TF.to_tensor(img1); img2 = TF.to_tensor(img2)
        img1 = TF.normalize(img1, self.mean, self.std)
        img2 = TF.normalize(img2, self.mean, self.std)
        return img1, img2

    def __getitem__(self, index: int):
        assert 0 <= index < len(self), f'index out of range: {index} / {len(self)}'

        img_path = self.lines[index]
        # front / side
        img1_path = img_path
        img2_path = Path(str(img_path).replace("front.jpg", "side.jpg"))

        img1_pil = Image.open(img1_path).convert('RGB')
        img2_pil = Image.open(img2_path).convert('RGB')

        if self.train and not self.test and self.transform is None:
            img1, img2 = self._pair_transform(img1_pil, img2_pil)
        else:
            img1  = self.transform(img1_pil)
            img2 = self.transform(img2_pil)

        class_name = Path(img_path).parent.parent.name
        label = self.cls_to_idx[class_name]

        return img1, img2, label

def byol_augment_batch(x):
    if torch.rand(1) < 0.5:
        x = torch.flip(x, dims=[3])
    if torch.rand(1) < 0.5:
        x = torch.flip(x, dims=[2])
    # color jitter (배치 단위)
    if torch.rand(1) < 0.8:
        factor = 1.0 + (torch.rand(1).item() * 0.4 - 0.2)
        x = torch.clamp(x * factor, 0, 1)
    return x

In [ ]:
# Part2. train/val/test 분할

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

train_root = r"/home/airl02/Desktop/ParkJeongEun/PediURF/train"
test_root = r"/home/airl02/Desktop/ParkJeongEun/PediURF/test"

cls_to_idx = {
    "Proximal ulna and radius fractures": 0,
    "Midshaft ulna and radius fractures": 1,
    "Distal ulna and radius fractures": 2,
}

def get_all_imgs_path(root: str):
  root = Path(root)
  all_front_path = [str(p) for p in root.rglob("front.jpg")]
  return all_front_path

def get_label_from_path(img_path: str):
  path = Path(img_path)
  class_name = path.parent.parent.name

  if class_name not in cls_to_idx:
    raise ValueError(f"Unknown class folder name: {class_name}")

  return cls_to_idx[class_name]

def make_stratified_split(train_root, val_ratio=0.2, seed=123):
  all_imgs_path = get_all_imgs_path(train_root)

  all_labels = [get_label_from_path(p) for p in all_imgs_path]

  train_imgs_path, val_imgs_path = train_test_split(
      all_imgs_path,
      test_size=val_ratio,
      random_state=seed,
      stratify=all_labels
  )

  return train_imgs_path, val_imgs_path

# 8:2 split
train_imgs_path, val_imgs_path = make_stratified_split(
    train_root = train_root,
    val_ratio=0.2,
    seed=seed
)

test_imgs_path = get_all_imgs_path(test_root)

train_data = Clsdata(train_imgs_path, cls_to_idx, train=True, shuffle=True)
val_data  = Clsdata(val_imgs_path, cls_to_idx, train=False, shuffle=False, test=True)
test_data = Clsdata(test_imgs_path, cls_to_idx, train=False, shuffle=False, test=True)

def seed_worker(worker_id):
    np.random.seed(seed + worker_id)
    random.seed(seed + worker_id)

g = torch.Generator()
g.manual_seed(seed)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                          num_workers=2, worker_init_fn=seed_worker, generator=g)
val_loader   = DataLoader(val_data, batch_size=batch_size, shuffle=False,
                          num_workers=2, worker_init_fn=seed_worker, generator=g)
test_loader  = DataLoader(test_data, batch_size=batch_size, shuffle=False,
                          num_workers=2, worker_init_fn=seed_worker, generator=g)

train_length = len(train_data)
val_length  = len(val_data)
test_length = len(test_data)

print(f"train_length: {train_length}")
print(f"val_length: {val_length}")
print(f"test_length: {test_length}")

In [ ]:
# Part3. Encoder (BYOL)

from torchvision.models import ResNet34_Weights

class MLP(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=1024, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, x):
        return self.net(x)

class BYOL(nn.Module):
    def __init__(self):
        super().__init__()
        self.online_encoder = models.resnet34(weights=ResNet34_Weights.DEFAULT)
        self.online_encoder.fc = nn.Identity()

        self.target_encoder = models.resnet34(weights=ResNet34_Weights.DEFAULT)
        self.target_encoder.fc = nn.Identity()

        self.online_projector = MLP()
        self.target_projector = MLP()
        self.predictor = MLP(in_dim=256)

        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_projector.parameters():
            p.requires_grad = False

    def forward(self, x1, x2):
        o1 = self.predictor(self.online_projector(self.online_encoder(x1)))
        o2 = self.predictor(self.online_projector(self.online_encoder(x2)))

        with torch.no_grad():
            t1 = self.target_projector(self.target_encoder(x1))
            t2 = self.target_projector(self.target_encoder(x2))

        return o1, o2, t1.detach(), t2.detach()

    @torch.no_grad()
    def update_target(self, tau=0.99):
        for o, t in zip(self.online_encoder.parameters(), self.target_encoder.parameters()):
            t.data = tau * t.data + (1 - tau) * o.data
        for o, t in zip(self.online_projector.parameters(), self.target_projector.parameters()):
            t.data = tau * t.data + (1 - tau) * o.data

byol_model = BYOL().to(device)

from torch.optim.lr_scheduler import StepLR

def byol_loss(p, z):
    return 2 - 2 * F.cosine_similarity(p, z, dim=-1).mean()

byol_optimizer = torch.optim.SGD(byol_model.parameters(), lr=b_learning_rate, momentum=0.9, weight_decay=1e-4)
byol_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(byol_optimizer, T_max=byol_epochs)

In [ ]:
# Part4. BYOL 학습

save_name = "byol_encoder2.pth"
encoder_save_dir = r"/home/airl02/Desktop/ParkJeongEun/Model"
os.makedirs(encoder_save_dir, exist_ok=True)

checkpoint_dir = os.path.join(encoder_save_dir, "byol_checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)

byol_losses = []

def get_latest_checkpoint(save_dir):
    files = os.listdir(save_dir)
    ckpts = [f for f in files if f.startswith("checkpoint_epoch_") and f.endswith(".pth")]
    if not ckpts:
        return None
    epochs = []
    for f in ckpts:
        stem = f.replace("checkpoint_epoch_", "").replace(".pth", "")
        if stem.isdigit():
            epochs.append((int(stem), f))
    if not epochs:
        return None
    max_epoch, max_file = max(epochs, key=lambda x: x[0])
    return os.path.join(save_dir, max_file)

checkpoint_path = get_latest_checkpoint(checkpoint_dir)

if checkpoint_path is not None:
    checkpoint = torch.load(checkpoint_path)
    byol_model.load_state_dict(checkpoint['model_state_dict'])
    byol_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    byol_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint['loss']
    byol_losses = checkpoint.get('losses', [])
    print(f"Resume from epoch {start_epoch}")
else:
    start_epoch = 0
    best_loss = float('inf')
    print("Start from scratch")

for epoch in range(start_epoch, byol_epochs):
    byol_model.train()
    total_loss = 0

    for img1, img2, _ in train_loader:
        img1 = img1.to(device)
        img2 = img2.to(device)

        byol_optimizer.zero_grad()

        # front/side 각각 augmentation (배치 단위)
        f1 = byol_augment_batch(img1)
        f2 = byol_augment_batch(img1)
        s1 = byol_augment_batch(img2)
        s2 = byol_augment_batch(img2)

        o1, _, _, t2 = byol_model(f1, s2)
        o2, _, _, t1 = byol_model(s1, f2)

        loss = byol_loss(o1, t2) + byol_loss(o2, t1)

        loss.backward()
        byol_optimizer.step()
        byol_model.update_target()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    byol_losses.append(avg_loss)
    byol_scheduler.step()

    print(f"[BYOL] Epoch {epoch+1}/{byol_epochs}, Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': byol_model.state_dict(),
            'optimizer_state_dict': byol_optimizer.state_dict(),
            'scheduler_state_dict': byol_scheduler.state_dict(),
            'loss': best_loss,
            'losses': byol_losses,
        }, os.path.join(checkpoint_dir, "best_model.pth"))

    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': byol_model.state_dict(),
            'optimizer_state_dict': byol_optimizer.state_dict(),
            'scheduler_state_dict': byol_scheduler.state_dict(),
            'loss': avg_loss,
            'losses': byol_losses,
        }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pth"))

# encoder 저장
byol_encoder_path = os.path.join(encoder_save_dir, save_name)
torch.save(byol_model.online_encoder.state_dict(), byol_encoder_path)

# loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(byol_losses) + 1), byol_losses)
plt.title("BYOL Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.show()